### TO DO
Criar variável de se houve queda ou não em relação ao ano passado <br>
Criar pipeline para testar Rando forest e regressão logistica <br>
Avaliar se é melhor fazer uma prediação de qual será a fase seguinte a avaliar se houve queda ou não
Tentar chegar em 75% de acurácia

# 0 Bibliotecas e Funções

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score 

In [2]:
def substituir_valores_coluna(
    df: pd.DataFrame,
    coluna: str,
    mapeamento: dict,
    converter_numerico: bool = True
) -> pd.DataFrame:
    """
    Substitui valores específicos de uma coluna usando um dicionário.

    Parâmetros
    ----------
    df : DataFrame
        DataFrame original
    coluna : str
        Nome da coluna a ser tratada
    mapeamento : dict
        Ex: {'baixo': 1, 'medio': 2, 'alto': 3}
    converter_numerico : bool
        Se True, tenta converter a coluna para numérico no final

    Retorno
    -------
    DataFrame com a coluna corrigida
    """

    df = df.copy()

    # Substitui apenas os valores especificados
    df[coluna] = df[coluna].replace(mapeamento)

    if converter_numerico:
        df[coluna] = pd.to_numeric(df[coluna], errors='coerce')

    return df


In [3]:
def corrigir_coluna_idade_com_datas(
    df: pd.DataFrame,
    coluna: str,
    idade_min: int = 3,
    idade_max: int = 120
) -> pd.DataFrame:
    """
    Substitui valores que são datas em uma coluna de idade
    pela média das idades válidas do DataFrame.

    Parâmetros
    ----------
    df : pd.DataFrame
        DataFrame original
    coluna : str
        Nome da coluna de idade
    idade_min : int
        Idade mínima válida
    idade_max : int
        Idade máxima válida

    Retorno
    -------
    DataFrame com a coluna corrigida
    """

    df = df.copy()

    # Tentativa de converter para numérico
    idade_num = pd.to_numeric(df[coluna], errors='coerce')

    # Tentativa de converter para data
    data_conv = pd.to_datetime(df[coluna], errors='coerce', dayfirst=True)

    # Idades válidas: número dentro do intervalo esperado
    idade_valida = idade_num.between(idade_min, idade_max)

    # Média das idades válidas
    media_idade = idade_num[idade_valida].mean()

    # Onde é data OU idade inválida → substituir pela média
    mask_substituir = (~idade_valida) & (data_conv.notna())

    df.loc[mask_substituir, coluna] = round(media_idade)

    # Converter coluna final para numérico
    df[coluna] = pd.to_numeric(df[coluna], errors='coerce')

    return df

# 1 Importacão de base

In [4]:
arquivo = r"https://raw.githubusercontent.com/vbomura/tc5/03b05a676fa48320170229c8be9c7a22fe5c0f7f/Codigos/base_anos_limpo.xlsx"

df = pd.read_excel(arquivo)

In [5]:
df.head(10)

,ra,fase,turma,nome_anonimizado,idade,genero,ano_ingresso,instituicao_de_ensino,pedra,inde,...,por,ing,ipv,ian,fase_ideal_original,defasagem,ano_aba,fase_ideal,ipp,fase_original
0,RA-1,7,A,Aluno-1,19,Feminino,2016,Escola Pública,Quartzo,5.783,...,3.5,6.0,7.278,5.0,Fase 8 (Universitários),-1,2022,FASE 8,NaN,NaN
1,RA-2,7,A,Aluno-2,17,Feminino,2017,Rede Decisão,Ametista,7.055,...,4.5,9.7,6.778,10.0,Fase 7 (3º EM),0,2022,FASE 7,NaN,NaN
2,RA-3,7,A,Aluno-3,17,Feminino,2016,Rede Decisão,Ágata,6.591,...,4.0,6.9,7.556,10.0,Fase 7 (3º EM),0,2022,FASE 7,NaN,NaN
3,RA-4,7,A,Aluno-4,17,Masculino,2017,Rede Decisão,Quartzo,5.951,...,3.5,8.7,5.278,10.0,Fase 7 (3º EM),0,2022,FASE 7,NaN,NaN
4,RA-5,7,A,Aluno-5,17,Feminino,2016,Rede Decisão,Ametista,7.427,...,2.9,5.7,7.389,10.0,Fase 7 (3º EM),0,2022,FASE 7,NaN,NaN
5,RA-6,7,A,Aluno-6,18,Feminino,2021,Escola Pública,Quartzo,5.848,...,5.3,2.3,7.222,5.0,Fase 8 (Universitários),-1,2022,FASE 8,NaN,NaN
6,RA-7,7,A,Aluno-7,18,Masculino,2017,Rede Decisão,Ágata,6.818,...,5.7,9.0,7.667,5.0,Fase 8 (Universitários),-1,2022,FASE 8,NaN,NaN
7,RA-8,7,A,Aluno-8,20,Feminino,2018,Escola Pública,Quartzo,4.786,...,0.7,2.9,6.278,5.0,Fase 8 (Universitários),-1,2022,FASE 8,NaN,NaN
8,RA-9,7,A,Aluno-9,18,Feminino,2019,Escola Pública,Topázio,8.109,...,6.0,8.7,9.500,5.0,Fase 8 (Universitários),-1,2022,FASE 8,NaN,NaN
9,RA-10,7,A,Aluno-10,18,Feminino,2021,Escola Pública,Quartzo,5.784,...,2.6,6.4,7.056,5.0,Fase 8 (Universitários),-1,2022,FASE 8,NaN,NaN


# 2 Criando a variável de queda

## Separando os casos de alunos que estiveram mais de um ano

In [6]:
# Separando alunos que ficarão mais de um ano no programa
alunos_repetidos = (
    df
    .groupby('ra')
    .size()
    .reset_index(name='qtd_registros')
)

alunos_repetidos = alunos_repetidos[
    alunos_repetidos['qtd_registros'] > 1
]

In [7]:
# Criando o dataframe e também já ordenando ele
df_repetidos = df[
    df['ra'].isin(alunos_repetidos['ra'])
].copy()

df_repetidos = df_repetidos.sort_values(
    by=['ra', 'ano_aba']
)

In [8]:
# Fazemos o shift, para podermos usar como target a mudança de fase
df_repetidos['defasagem_prox_ano'] = (
    df_repetidos
    .groupby('ra')['defasagem']
    .shift(-1)
)

# Cria uma variável para identificar se houve queda ou não
df_repetidos['movimento_fase'] = np.sign(
    df_repetidos['defasagem_prox_ano'] - df_repetidos['defasagem']
)

df_modelo = df_repetidos.dropna(subset=['movimento_fase']).copy()

## Pipeline

## Ajustes nas bases

In [9]:
# Corrigindo valores das fases
mapeamento_defasagem = {
    'ALFA': 0,
    'FASE 1': 1,
    'FASE 2': 2,
    'FASE 3': 3,
    'FASE 4': 4,
    'FASE 5': 5,
    'FASE 6': 6 
}

df_modelo = substituir_valores_coluna(
    df=df_modelo,
    coluna='fase',
    mapeamento=mapeamento_defasagem
)

# Ajustando os valores da target por causa do regressão logístico
mapeamento_target = {
    -1: 0,  # queda
     0: 1,  # mantém
     1: 2   # aumento
}

df_modelo = substituir_valores_coluna(
    df=df_modelo,
    coluna='defasagem_prox_ano',
    mapeamento=mapeamento_defasagem
)

# Corrigindo os valores de idade (fazemos uma media para os dados vazios)
df_modelo = corrigir_coluna_idade_com_datas(
    df=df_modelo,
    coluna='idade'
)


C:\Users\bryan\AppData\Local\Temp\ipykernel_24904\3350390966.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[coluna] = df[coluna].replace(mapeamento)


In [10]:
# Algumas variáveis são textos, vamos tratar com one-hot encoder
categoricas = ['genero', 'instituicao_de_ensino', 'pedra']

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(
            drop='first',
            handle_unknown='ignore'
        ), categoricas)
    ]
)

## Previsao

In [11]:
# Definidndo a target
TARGET = 'movimento_fase'

# Definindo as features
FEATURES = [
            'ra',
            'genero', 
            'inde', 
            'iaa', 
            'ieg', 
            'ips', 
            'ida', 
            'mat', 
            'por', 
            'ipv', 
            'ian', 
            'defasagem',
            'ipp',
            'instituicao_de_ensino', 
            'pedra', 
            'fase'
            ]


X = df_modelo[FEATURES]
y = df_modelo[TARGET]


In [12]:
# Separando teste e treino
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

# Vai evitar que o mesmo aluno no mesmo treino
train_idx, test_idx = next(
    gss.split(X, y, groups=df_modelo['ra'])
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


In [13]:
# Regressão logística
pipe_log = Pipeline(steps=[
    ('prep', preprocessador),
    ('scaler', StandardScaler(with_mean=False)),
    ('model', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        multi_class='ovr',
        solver='lbfgs',
        random_state=42
    ))
])

# Random Forest
pipe_rf = Pipeline(steps=[
    ('prep', preprocessador),
    ('model', RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=20,
        class_weight='balanced',
        random_state=42
    ))
])

In [14]:
pipe_log.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)

y_pred_log = pipe_log.predict(X_test)
y_pred_rf = pipe_rf.predict(X_test)

y_prob_log = pipe_log.predict(X_test)
y_prob_rf = pipe_rf.predict(X_test)

c:\Users\bryan\anaconda3\envs\fase4\Lib\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


c:\Users\bryan\anaconda3\envs\fase4\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\bryan\anaconda3\envs\fase4\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\bryan\anaconda3\envs\fase4\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\bryan\anaconda3\envs\fase4\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


## Avaliando os modelos

In [15]:
print('Regressão Logística')
print(classification_report(y_test, y_pred_log))
print('ROC AUC:', f1_score(y_test, y_prob_log, average='macro'))

print('\nRandom Forest')
print(classification_report(y_test, y_pred_rf))
print('ROC AUC:', f1_score(y_test, y_prob_rf, average='macro'))

Regressão Logística
              precision    recall  f1-score   support

        -1.0       0.24      0.19      0.21        37
         0.0       0.53      0.53      0.53       107
         1.0       0.50      0.53      0.52       105

    accuracy                           0.48       249
   macro avg       0.42      0.42      0.42       249
weighted avg       0.47      0.48      0.48       249

ROC AUC: 0.4194942675062705

Random Forest
              precision    recall  f1-score   support

        -1.0       0.16      0.32      0.22        37
         0.0       0.47      0.34      0.39       107
         1.0       0.45      0.43      0.44       105

    accuracy                           0.37       249
   macro avg       0.36      0.36      0.35       249
weighted avg       0.42      0.37      0.39       249

ROC AUC: 0.35022087886538017


# Testes